In [7]:
import torch
print(torch.__version__)

2.5.1+cu118


In [8]:
!pip install torch==2.6.0 torchvision torchaudio

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 945.7 kB/s  0:13:17:00:0100:15m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.2 MB/s  0:04:15m0:00:0100:07m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 2.3 MB/s  0:00:05 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 1.3 MB/s  0:00:18m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 781.8 kB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.2 MB/s  0:04:18m0:00:0100:04m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.7 MB/s  0:01:42m0:00:0100:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 2.2 MB/s  0:00:25m0:

## 1. Speech Model Training 1(xlsr)

In [1]:
import torch
import torchaudio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, confusion_matrix
from dataclasses import dataclass
from typing import Any, Dict, List

# ----------------------------------------
# 1. Language Labels
# ----------------------------------------

languages = [
    "bengali","gujarati","hindi","kannada","malayalam",
    "marathi","odia","punjabi","sanskrit","tamil","telugu","urdu"
]

label2id = {l:i for i,l in enumerate(languages)}
id2label = {i:l for i,l in enumerate(languages)}

MODEL_NAME = "facebook/wav2vec2-large-xlsr-53"

# ----------------------------------------
# 2. Feature Extractor
# ----------------------------------------

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)

# ----------------------------------------
# 3. Dataset Class
# ----------------------------------------

class SpeechDataset(Dataset):

    def __init__(self, csv_file):

        self.data = pd.read_csv(csv_file)

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        path = row["file_path"]
        label = label2id[row["language"]]

        speech, sr = torchaudio.load(path)

        speech = speech.squeeze()

        inputs = feature_extractor(
            speech,
            sampling_rate=16000,
            return_tensors="pt"
        )

        return {
            "input_values": inputs.input_values.squeeze(),
            "labels": torch.tensor(label)
        }

# ----------------------------------------
# 4. Dynamic Padding
# ----------------------------------------

@dataclass
class DataCollatorWithPadding:

    feature_extractor: Any

    def __call__(self, features: List[Dict[str, Any]]):

        input_features = [{"input_values": f["input_values"]} for f in features]

        batch = self.feature_extractor.pad(
            input_features,
            padding=True,
            return_tensors="pt"
        )

        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)

        batch["labels"] = labels

        return batch


data_collator = DataCollatorWithPadding(feature_extractor)

# ----------------------------------------
# 5. Load Datasets
# ----------------------------------------

train_dataset = SpeechDataset("master_valid_augmented_cleaned.csv")

val_dataset = SpeechDataset(
    "master_test_known_dataset_standardized2.csv"
)

test_dataset = SpeechDataset(
    "master_test_unknown_dataset_standardized2.csv"
)

# ----------------------------------------
# 6. Load Model
# ----------------------------------------

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=12,
    label2id=label2id,
    id2label=id2label,
    use_safetensors=True
)

model.freeze_feature_encoder()

# ----------------------------------------
# 7. Training Configuration
# ----------------------------------------

training_args = TrainingArguments(

    output_dir="./results_xlsr",

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    eval_strategy="epoch",

    num_train_epochs=50,

    learning_rate=2e-5,

    gradient_accumulation_steps=4,

    fp16=True,

    logging_steps=100,

    save_total_limit=2
)

# ----------------------------------------
# 8. Evaluation Metric
# ----------------------------------------

def compute_metrics(pred):

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(pred.label_ids, preds)
    }

# ----------------------------------------
# 9. Trainer
# ----------------------------------------

trainer = Trainer(

    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

# ----------------------------------------
# 10. Train Model
# ----------------------------------------

print("🚀 Starting Speech Model Training...")

trainer.train()

# ----------------------------------------
# 11. Plot Training Loss Curve
# ----------------------------------------

logs = trainer.state.log_history

train_loss = []
eval_loss = []
epochs = []

for log in logs:

    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
        epochs.append(log["epoch"])

    if "eval_loss" in log:
        eval_loss.append(log["eval_loss"])

plt.figure(figsize=(8,5))
plt.plot(train_loss, label="Training Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.legend()

plt.savefig("results_xlsr/loss_curve.png")
plt.show()

# ----------------------------------------
# 12. Accuracy Curve
# ----------------------------------------

eval_accuracy = []
eval_epochs = []

for log in logs:

    if "eval_accuracy" in log:
        eval_accuracy.append(log["eval_accuracy"])
        eval_epochs.append(log["epoch"])

plt.figure(figsize=(8,5))
plt.plot(eval_epochs, eval_accuracy, marker="o")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Curve")

plt.savefig("results_xlsr/accuracy_curve.png")
plt.show()

# ----------------------------------------
# 13. Final Evaluation
# ----------------------------------------

print("\n🔥 Evaluating on Test Dataset...")

results = trainer.predict(test_dataset)

y_true = results.label_ids
y_pred = results.predictions.argmax(-1)

print("Final Test Accuracy:", accuracy_score(y_true, y_pred))

# ----------------------------------------
# 14. Confusion Matrix
# ----------------------------------------

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=languages,
    yticklabels=languages
)

plt.xlabel("Predicted Language")
plt.ylabel("True Language")
plt.title("Confusion Matrix")

plt.savefig("results_xlsr/confusion_matrix.png")

plt.show()

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-large-xlsr-53 and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🚀 Starting Speech Model Training...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## 2. Speech Model Training 2 (hubert-large-ls960-ft)

In [ ]:
import torch
import torchaudio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, confusion_matrix
from dataclasses import dataclass
from typing import Any, Dict, List

# ----------------------------------------
# 1. Language Labels
# ----------------------------------------

languages = [
    "bengali","gujarati","hindi","kannada","malayalam",
    "marathi","odia","punjabi","sanskrit","tamil","telugu","urdu"
]

label2id = {l:i for i,l in enumerate(languages)}
id2label = {i:l for i,l in enumerate(languages)}

MODEL_NAME = "facebook/hubert-large-ls960-ft"

# ----------------------------------------
# 2. Feature Extractor
# ----------------------------------------

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)

# ----------------------------------------
# 3. Dataset Class
# ----------------------------------------

class SpeechDataset(Dataset):

    def __init__(self, csv_file):

        self.data = pd.read_csv(csv_file)

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        path = row["file_path"]
        label = label2id[row["language"]]

        speech, sr = torchaudio.load(path)

        speech = speech.squeeze()

        inputs = feature_extractor(
            speech,
            sampling_rate=16000,
            return_tensors="pt"
        )

        return {
            "input_values": inputs.input_values.squeeze(),
            "labels": torch.tensor(label)
        }

# ----------------------------------------
# 4. Dynamic Padding
# ----------------------------------------

@dataclass
class DataCollatorWithPadding:

    feature_extractor: Any

    def __call__(self, features: List[Dict[str, Any]]):

        input_features = [{"input_values": f["input_values"]} for f in features]

        batch = self.feature_extractor.pad(
            input_features,
            padding=True,
            return_tensors="pt"
        )

        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)

        batch["labels"] = labels

        return batch


data_collator = DataCollatorWithPadding(feature_extractor)

# ----------------------------------------
# 5. Load Datasets
# ----------------------------------------

train_dataset = SpeechDataset("master_valid_augmented_cleaned.csv")

val_dataset = SpeechDataset(
    "master_test_known_dataset_standardized2.csv"
)

test_dataset = SpeechDataset(
    "master_test_unknown_dataset_standardized2.csv"
)

# ----------------------------------------
# 6. Load Model
# ----------------------------------------

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=12,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
    use_safetensors=True
)

model.freeze_feature_encoder()

# ----------------------------------------
# 7. Training Configuration
# ----------------------------------------

training_args = TrainingArguments(

    output_dir="./speech_results_hubert",

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    eval_strategy="epoch",

    num_train_epochs=50,

    learning_rate=2e-5,

    gradient_accumulation_steps=4,

    fp16=True,

    logging_steps=100,

    save_total_limit=2
)

# ----------------------------------------
# 8. Evaluation Metric
# ----------------------------------------

def compute_metrics(pred):

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(pred.label_ids, preds)
    }

# ----------------------------------------
# 9. Trainer
# ----------------------------------------

trainer = Trainer(

    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

# ----------------------------------------
# 10. Train Model
# ----------------------------------------

print("🚀 Starting Speech Model Training...")

trainer.train()

# ----------------------------------------
# 11. Plot Training Loss Curve
# ----------------------------------------

logs = trainer.state.log_history

train_loss = []
eval_loss = []
epochs = []

for log in logs:

    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
        epochs.append(log["epoch"])

    if "eval_loss" in log:
        eval_loss.append(log["eval_loss"])

plt.figure(figsize=(8,5))
plt.plot(train_loss, label="Training Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.legend()

plt.savefig("speech_results_hubert/loss_curve.png")
plt.show()

# ----------------------------------------
# 12. Accuracy Curve
# ----------------------------------------

eval_accuracy = []
eval_epochs = []

for log in logs:

    if "eval_accuracy" in log:
        eval_accuracy.append(log["eval_accuracy"])
        eval_epochs.append(log["epoch"])

plt.figure(figsize=(8,5))
plt.plot(eval_epochs, eval_accuracy, marker="o")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Curve")

plt.savefig("speech_results_hubert/accuracy_curve.png")
plt.show()

# ----------------------------------------
# 13. Final Evaluation
# ----------------------------------------

print("\n🔥 Evaluating on Test Dataset...")

results = trainer.predict(test_dataset)

y_true = results.label_ids
y_pred = results.predictions.argmax(-1)

print("Final Test Accuracy:", accuracy_score(y_true, y_pred))

# ----------------------------------------
# 14. Confusion Matrix
# ----------------------------------------

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=languages,
    yticklabels=languages
)

plt.xlabel("Predicted Language")
plt.ylabel("True Language")
plt.title("Confusion Matrix")

plt.savefig("speech_results_hubert/confusion_matrix.png")

plt.show()

## 3. Speech Model Training3 (wavlm-base)

In [ ]:
import torch
import torchaudio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, confusion_matrix
from dataclasses import dataclass
from typing import Any, Dict, List

# ----------------------------------------
# 1. Language Labels
# ----------------------------------------

languages = [
    "bengali","gujarati","hindi","kannada","malayalam",
    "marathi","odia","punjabi","sanskrit","tamil","telugu","urdu"
]

label2id = {l:i for i,l in enumerate(languages)}
id2label = {i:l for i,l in enumerate(languages)}

MODEL_NAME = "microsoft/wavlm-base"

# ----------------------------------------
# 2. Feature Extractor
# ----------------------------------------

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)

# ----------------------------------------
# 3. Dataset Class
# ----------------------------------------

class SpeechDataset(Dataset):

    def __init__(self, csv_file):

        self.data = pd.read_csv(csv_file)

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        path = row["file_path"]
        label = label2id[row["language"]]

        speech, sr = torchaudio.load(path)

        speech = speech.squeeze()

        inputs = feature_extractor(
            speech,
            sampling_rate=16000,
            return_tensors="pt"
        )

        return {
            "input_values": inputs.input_values.squeeze(),
            "labels": torch.tensor(label)
        }

# ----------------------------------------
# 4. Dynamic Padding
# ----------------------------------------

@dataclass
class DataCollatorWithPadding:

    feature_extractor: Any

    def __call__(self, features: List[Dict[str, Any]]):

        input_features = [{"input_values": f["input_values"]} for f in features]

        batch = self.feature_extractor.pad(
            input_features,
            padding=True,
            return_tensors="pt"
        )

        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)

        batch["labels"] = labels

        return batch


data_collator = DataCollatorWithPadding(feature_extractor)

# ----------------------------------------
# 5. Load Datasets
# ----------------------------------------

train_dataset = SpeechDataset("master_valid_augmented_cleaned.csv")

val_dataset = SpeechDataset(
    "master_test_known_dataset_standardized2.csv"
)

test_dataset = SpeechDataset(
    "master_test_unknown_dataset_standardized2.csv"
)

# ----------------------------------------
# 6. Load Model
# ----------------------------------------

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=12,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
    use_safetensors=True
)

model.freeze_feature_encoder()

# ----------------------------------------
# 7. Training Configuration
# ----------------------------------------

training_args = TrainingArguments(

    output_dir="./speech_results_wavlm",

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    eval_strategy="epoch",

    num_train_epochs=50,

    learning_rate=2e-5,

    gradient_accumulation_steps=4,

    fp16=True,

    logging_steps=100,

    save_total_limit=2
)

# ----------------------------------------
# 8. Evaluation Metric
# ----------------------------------------

def compute_metrics(pred):

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(pred.label_ids, preds)
    }

# ----------------------------------------
# 9. Trainer
# ----------------------------------------

trainer = Trainer(

    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

# ----------------------------------------
# 10. Train Model
# ----------------------------------------

print("🚀 Starting Speech Model Training...")

trainer.train()

# ----------------------------------------
# 11. Plot Training Loss Curve
# ----------------------------------------

logs = trainer.state.log_history

train_loss = []
eval_loss = []
epochs = []

for log in logs:

    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
        epochs.append(log["epoch"])

    if "eval_loss" in log:
        eval_loss.append(log["eval_loss"])

plt.figure(figsize=(8,5))
plt.plot(train_loss, label="Training Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.legend()

plt.savefig("speech_results_wavlm/loss_curve.png")
plt.show()

# ----------------------------------------
# 12. Accuracy Curve
# ----------------------------------------

eval_accuracy = []
eval_epochs = []

for log in logs:

    if "eval_accuracy" in log:
        eval_accuracy.append(log["eval_accuracy"])
        eval_epochs.append(log["epoch"])

plt.figure(figsize=(8,5))
plt.plot(eval_epochs, eval_accuracy, marker="o")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Curve")

plt.savefig("speech_results_wavlm/accuracy_curve.png")
plt.show()

# ----------------------------------------
# 13. Final Evaluation
# ----------------------------------------

print("\n🔥 Evaluating on Test Dataset...")

results = trainer.predict(test_dataset)

y_true = results.label_ids
y_pred = results.predictions.argmax(-1)

print("Final Test Accuracy:", accuracy_score(y_true, y_pred))

# ----------------------------------------
# 14. Confusion Matrix
# ----------------------------------------

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=languages,
    yticklabels=languages
)

plt.xlabel("Predicted Language")
plt.ylabel("True Language")
plt.title("Confusion Matrix")

plt.savefig("speech_results_wavlm/confusion_matrix.png")

plt.show()

## 4. Speech Model Training 4 (wav2vec2-base)

In [ ]:
import torch
import torchaudio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, confusion_matrix
from dataclasses import dataclass
from typing import Any, Dict, List

# ----------------------------------------
# 1. Language Labels
# ----------------------------------------

languages = [
    "bengali","gujarati","hindi","kannada","malayalam",
    "marathi","odia","punjabi","sanskrit","tamil","telugu","urdu"
]

label2id = {l:i for i,l in enumerate(languages)}
id2label = {i:l for i,l in enumerate(languages)}

MODEL_NAME = "facebook/wav2vec2-base"

# ----------------------------------------
# 2. Feature Extractor
# ----------------------------------------

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)

# ----------------------------------------
# 3. Dataset Class
# ----------------------------------------

class SpeechDataset(Dataset):

    def __init__(self, csv_file):

        self.data = pd.read_csv(csv_file)

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        path = row["file_path"]
        label = label2id[row["language"]]

        speech, sr = torchaudio.load(path)

        speech = speech.squeeze()

        inputs = feature_extractor(
            speech,
            sampling_rate=16000,
            return_tensors="pt"
        )

        return {
            "input_values": inputs.input_values.squeeze(),
            "labels": torch.tensor(label)
        }

# ----------------------------------------
# 4. Dynamic Padding
# ----------------------------------------

@dataclass
class DataCollatorWithPadding:

    feature_extractor: Any

    def __call__(self, features: List[Dict[str, Any]]):

        input_features = [{"input_values": f["input_values"]} for f in features]

        batch = self.feature_extractor.pad(
            input_features,
            padding=True,
            return_tensors="pt"
        )

        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)

        batch["labels"] = labels

        return batch


data_collator = DataCollatorWithPadding(feature_extractor)

# ----------------------------------------
# 5. Load Datasets
# ----------------------------------------

train_dataset = SpeechDataset("master_valid_augmented_cleaned.csv")

val_dataset = SpeechDataset(
    "master_test_known_dataset_standardized2.csv"
)

test_dataset = SpeechDataset(
    "master_test_unknown_dataset_standardized2.csv"
)

# ----------------------------------------
# 6. Load Model
# ----------------------------------------

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=12,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
    use_safetensors=True
)

model.freeze_feature_encoder()

# ----------------------------------------
# 7. Training Configuration
# ----------------------------------------

training_args = TrainingArguments(

    output_dir="./speech_results_wav2vec2_base",

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    eval_strategy="epoch",

    num_train_epochs=50,

    learning_rate=2e-5,

    gradient_accumulation_steps=4,

    fp16=True,

    logging_steps=100,

    save_total_limit=2
)

# ----------------------------------------
# 8. Evaluation Metric
# ----------------------------------------

def compute_metrics(pred):

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(pred.label_ids, preds)
    }

# ----------------------------------------
# 9. Trainer
# ----------------------------------------

trainer = Trainer(

    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

# ----------------------------------------
# 10. Train Model
# ----------------------------------------

print("🚀 Starting Speech Model Training...")

trainer.train()

# ----------------------------------------
# 11. Plot Training Loss Curve
# ----------------------------------------

logs = trainer.state.log_history

train_loss = []
eval_loss = []
epochs = []

for log in logs:

    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
        epochs.append(log["epoch"])

    if "eval_loss" in log:
        eval_loss.append(log["eval_loss"])

plt.figure(figsize=(8,5))
plt.plot(train_loss, label="Training Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.legend()

plt.savefig("speech_results_wav2vec2_base/loss_curve.png")
plt.show()

# ----------------------------------------
# 12. Accuracy Curve
# ----------------------------------------

eval_accuracy = []
eval_epochs = []

for log in logs:

    if "eval_accuracy" in log:
        eval_accuracy.append(log["eval_accuracy"])
        eval_epochs.append(log["epoch"])

plt.figure(figsize=(8,5))
plt.plot(eval_epochs, eval_accuracy, marker="o")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Curve")

plt.savefig("speech_results_wav2vec2_base/accuracy_curve.png")
plt.show()

# ----------------------------------------
# 13. Final Evaluation
# ----------------------------------------

print("\n🔥 Evaluating on Test Dataset...")

results = trainer.predict(test_dataset)

y_true = results.label_ids
y_pred = results.predictions.argmax(-1)

print("Final Test Accuracy:", accuracy_score(y_true, y_pred))

# ----------------------------------------
# 14. Confusion Matrix
# ----------------------------------------

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=languages,
    yticklabels=languages
)

plt.xlabel("Predicted Language")
plt.ylabel("True Language")
plt.title("Confusion Matrix")

plt.savefig("speech_results_wav2vec2_base/confusion_matrix.png")

plt.show()

## 5. Speech Model Training 5 (wav2vec2-large-960h)

In [ ]:
import torch
import torchaudio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, confusion_matrix
from dataclasses import dataclass
from typing import Any, Dict, List

# ----------------------------------------
# 1. Language Labels
# ----------------------------------------

languages = [
    "bengali","gujarati","hindi","kannada","malayalam",
    "marathi","odia","punjabi","sanskrit","tamil","telugu","urdu"
]

label2id = {l:i for i,l in enumerate(languages)}
id2label = {i:l for i,l in enumerate(languages)}

MODEL_NAME = "facebook/wav2vec2-large-960h"

# ----------------------------------------
# 2. Feature Extractor
# ----------------------------------------

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)

# ----------------------------------------
# 3. Dataset Class
# ----------------------------------------

class SpeechDataset(Dataset):

    def __init__(self, csv_file):

        self.data = pd.read_csv(csv_file)

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        path = row["file_path"]
        label = label2id[row["language"]]

        speech, sr = torchaudio.load(path)

        speech = speech.squeeze()

        inputs = feature_extractor(
            speech,
            sampling_rate=16000,
            return_tensors="pt"
        )

        return {
            "input_values": inputs.input_values.squeeze(),
            "labels": torch.tensor(label)
        }

# ----------------------------------------
# 4. Dynamic Padding
# ----------------------------------------

@dataclass
class DataCollatorWithPadding:

    feature_extractor: Any

    def __call__(self, features: List[Dict[str, Any]]):

        input_features = [{"input_values": f["input_values"]} for f in features]

        batch = self.feature_extractor.pad(
            input_features,
            padding=True,
            return_tensors="pt"
        )

        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)

        batch["labels"] = labels

        return batch


data_collator = DataCollatorWithPadding(feature_extractor)

# ----------------------------------------
# 5. Load Datasets
# ----------------------------------------

train_dataset = SpeechDataset("master_valid_augmented_cleaned.csv")

val_dataset = SpeechDataset(
    "master_test_known_dataset_standardized2.csv"
)

test_dataset = SpeechDataset(
    "master_test_unknown_dataset_standardized2.csv"
)

# ----------------------------------------
# 6. Load Model
# ----------------------------------------

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=12,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
    use_safetensors=True
)

model.freeze_feature_encoder()

# ----------------------------------------
# 7. Training Configuration
# ----------------------------------------

training_args = TrainingArguments(

    output_dir="./speech_results_wav2vec2_large",

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    eval_strategy="epoch",

    num_train_epochs=50,

    learning_rate=2e-5,

    gradient_accumulation_steps=4,

    fp16=True,

    logging_steps=100,

    save_total_limit=2
)

# ----------------------------------------
# 8. Evaluation Metric
# ----------------------------------------

def compute_metrics(pred):

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(pred.label_ids, preds)
    }

# ----------------------------------------
# 9. Trainer
# ----------------------------------------

trainer = Trainer(

    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

# ----------------------------------------
# 10. Train Model
# ----------------------------------------

print("🚀 Starting Speech Model Training...")

trainer.train()

# ----------------------------------------
# 11. Plot Training Loss Curve
# ----------------------------------------

logs = trainer.state.log_history

train_loss = []
eval_loss = []
epochs = []

for log in logs:

    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
        epochs.append(log["epoch"])

    if "eval_loss" in log:
        eval_loss.append(log["eval_loss"])

plt.figure(figsize=(8,5))
plt.plot(train_loss, label="Training Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.legend()

plt.savefig("speech_results_wav2vec2_large/loss_curve.png")
plt.show()

# ----------------------------------------
# 12. Accuracy Curve
# ----------------------------------------

eval_accuracy = []
eval_epochs = []

for log in logs:

    if "eval_accuracy" in log:
        eval_accuracy.append(log["eval_accuracy"])
        eval_epochs.append(log["epoch"])

plt.figure(figsize=(8,5))
plt.plot(eval_epochs, eval_accuracy, marker="o")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Curve")

plt.savefig("speech_results_wav2vec2_large/accuracy_curve.png")
plt.show()

# ----------------------------------------
# 13. Final Evaluation
# ----------------------------------------

print("\n🔥 Evaluating on Test Dataset...")

results = trainer.predict(test_dataset)

y_true = results.label_ids
y_pred = results.predictions.argmax(-1)

print("Final Test Accuracy:", accuracy_score(y_true, y_pred))

# ----------------------------------------
# 14. Confusion Matrix
# ----------------------------------------

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=languages,
    yticklabels=languages
)

plt.xlabel("Predicted Language")
plt.ylabel("True Language")
plt.title("Confusion Matrix")

plt.savefig("speech_results_wav2vec2_large/confusion_matrix.png")

plt.show()

## 6. Speech Model Training 6 (wav2vec2-large-robust)

In [ ]:
import torch
import torchaudio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, confusion_matrix
from dataclasses import dataclass
from typing import Any, Dict, List

# ----------------------------------------
# 1. Language Labels
# ----------------------------------------

languages = [
    "bengali","gujarati","hindi","kannada","malayalam",
    "marathi","odia","punjabi","sanskrit","tamil","telugu","urdu"
]

label2id = {l:i for i,l in enumerate(languages)}
id2label = {i:l for i,l in enumerate(languages)}

MODEL_NAME = "facebook/wav2vec2-large-robust"

# ----------------------------------------
# 2. Feature Extractor
# ----------------------------------------

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)

# ----------------------------------------
# 3. Dataset Class
# ----------------------------------------

class SpeechDataset(Dataset):

    def __init__(self, csv_file):

        self.data = pd.read_csv(csv_file)

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        path = row["file_path"]
        label = label2id[row["language"]]

        speech, sr = torchaudio.load(path)

        speech = speech.squeeze()

        inputs = feature_extractor(
            speech,
            sampling_rate=16000,
            return_tensors="pt"
        )

        return {
            "input_values": inputs.input_values.squeeze(),
            "labels": torch.tensor(label)
        }

# ----------------------------------------
# 4. Dynamic Padding
# ----------------------------------------

@dataclass
class DataCollatorWithPadding:

    feature_extractor: Any

    def __call__(self, features: List[Dict[str, Any]]):

        input_features = [{"input_values": f["input_values"]} for f in features]

        batch = self.feature_extractor.pad(
            input_features,
            padding=True,
            return_tensors="pt"
        )

        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)

        batch["labels"] = labels

        return batch


data_collator = DataCollatorWithPadding(feature_extractor)

# ----------------------------------------
# 5. Load Datasets
# ----------------------------------------

train_dataset = SpeechDataset("master_valid_augmented_cleaned.csv")

val_dataset = SpeechDataset(
    "master_test_known_dataset_standardized2.csv"
)

test_dataset = SpeechDataset(
    "master_test_unknown_dataset_standardized2.csv"
)

# ----------------------------------------
# 6. Load Model
# ----------------------------------------

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=12,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
    use_safetensors=True
)

model.freeze_feature_encoder()

# ----------------------------------------
# 7. Training Configuration
# ----------------------------------------

training_args = TrainingArguments(

    output_dir="./speech_results_speechstew",

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    eval_strategy="epoch",

    num_train_epochs=50,

    learning_rate=2e-5,

    gradient_accumulation_steps=4,

    fp16=True,

    logging_steps=100,

    save_total_limit=2
)

# ----------------------------------------
# 8. Evaluation Metric
# ----------------------------------------

def compute_metrics(pred):

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(pred.label_ids, preds)
    }

# ----------------------------------------
# 9. Trainer
# ----------------------------------------

trainer = Trainer(

    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

# ----------------------------------------
# 10. Train Model
# ----------------------------------------

print("🚀 Starting Speech Model Training...")

trainer.train()

# ----------------------------------------
# 11. Plot Training Loss Curve
# ----------------------------------------

logs = trainer.state.log_history

train_loss = []
eval_loss = []
epochs = []

for log in logs:

    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
        epochs.append(log["epoch"])

    if "eval_loss" in log:
        eval_loss.append(log["eval_loss"])

plt.figure(figsize=(8,5))
plt.plot(train_loss, label="Training Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.legend()

plt.savefig("speech_results_speechstew/loss_curve.png")
plt.show()

# ----------------------------------------
# 12. Accuracy Curve
# ----------------------------------------

eval_accuracy = []
eval_epochs = []

for log in logs:

    if "eval_accuracy" in log:
        eval_accuracy.append(log["eval_accuracy"])
        eval_epochs.append(log["epoch"])

plt.figure(figsize=(8,5))
plt.plot(eval_epochs, eval_accuracy, marker="o")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Curve")

plt.savefig("speech_results_speechstew/accuracy_curve.png")
plt.show()

# ----------------------------------------
# 13. Final Evaluation
# ----------------------------------------

print("\n🔥 Evaluating on Test Dataset...")

results = trainer.predict(test_dataset)

y_true = results.label_ids
y_pred = results.predictions.argmax(-1)

print("Final Test Accuracy:", accuracy_score(y_true, y_pred))

# ----------------------------------------
# 14. Confusion Matrix
# ----------------------------------------

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=languages,
    yticklabels=languages
)

plt.xlabel("Predicted Language")
plt.ylabel("True Language")
plt.title("Confusion Matrix")

plt.savefig("speech_results_speechstew/confusion_matrix.png")

plt.show()

## 7. Speech Model Training 7 (wav2vec2-large-xlsr-53-english)

In [ ]:
import torch
import torchaudio
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, confusion_matrix
from dataclasses import dataclass
from typing import Any, Dict, List

# ----------------------------------------
# 1. Language Labels
# ----------------------------------------

languages = [
    "bengali","gujarati","hindi","kannada","malayalam",
    "marathi","odia","punjabi","sanskrit","tamil","telugu","urdu"
]

label2id = {l:i for i,l in enumerate(languages)}
id2label = {i:l for i,l in enumerate(languages)}

MODEL_NAME = "jonatasgrosman/wav2vec2-large-xlsr-53-english"

# ----------------------------------------
# 2. Feature Extractor
# ----------------------------------------

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)

# ----------------------------------------
# 3. Dataset Class
# ----------------------------------------

class SpeechDataset(Dataset):

    def __init__(self, csv_file):

        self.data = pd.read_csv(csv_file)

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        path = row["file_path"]
        label = label2id[row["language"]]

        speech, sr = torchaudio.load(path)

        speech = speech.squeeze()

        inputs = feature_extractor(
            speech,
            sampling_rate=16000,
            return_tensors="pt"
        )

        return {
            "input_values": inputs.input_values.squeeze(),
            "labels": torch.tensor(label)
        }

# ----------------------------------------
# 4. Dynamic Padding
# ----------------------------------------

@dataclass
class DataCollatorWithPadding:

    feature_extractor: Any

    def __call__(self, features: List[Dict[str, Any]]):

        input_features = [{"input_values": f["input_values"]} for f in features]

        batch = self.feature_extractor.pad(
            input_features,
            padding=True,
            return_tensors="pt"
        )

        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)

        batch["labels"] = labels

        return batch


data_collator = DataCollatorWithPadding(feature_extractor)

# ----------------------------------------
# 5. Load Datasets
# ----------------------------------------

train_dataset = SpeechDataset("master_valid_augmented_cleaned.csv")

val_dataset = SpeechDataset(
    "master_test_known_dataset_standardized2.csv"
)

test_dataset = SpeechDataset(
    "master_test_unknown_dataset_standardized2.csv"
)

# ----------------------------------------
# 6. Load Model
# ----------------------------------------

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=12,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
    use_safetensors=True
)

model.freeze_feature_encoder()

# ----------------------------------------
# 7. Training Configuration
# ----------------------------------------

training_args = TrainingArguments(

    output_dir="./speech_results_xlsr_varian",

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    eval_strategy="epoch",

    num_train_epochs=50,

    learning_rate=2e-5,

    gradient_accumulation_steps=4,

    fp16=True,

    logging_steps=100,

    save_total_limit=2
)

# ----------------------------------------
# 8. Evaluation Metric
# ----------------------------------------

def compute_metrics(pred):

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(pred.label_ids, preds)
    }

# ----------------------------------------
# 9. Trainer
# ----------------------------------------

trainer = Trainer(

    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

# ----------------------------------------
# 10. Train Model
# ----------------------------------------

print("🚀 Starting Speech Model Training...")

trainer.train()

# ----------------------------------------
# 11. Plot Training Loss Curve
# ----------------------------------------

logs = trainer.state.log_history

train_loss = []
eval_loss = []
epochs = []

for log in logs:

    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
        epochs.append(log["epoch"])

    if "eval_loss" in log:
        eval_loss.append(log["eval_loss"])

plt.figure(figsize=(8,5))
plt.plot(train_loss, label="Training Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.legend()

plt.savefig("speech_results_xlsr_varian/loss_curve.png")
plt.show()

# ----------------------------------------
# 12. Accuracy Curve
# ----------------------------------------

eval_accuracy = []
eval_epochs = []

for log in logs:

    if "eval_accuracy" in log:
        eval_accuracy.append(log["eval_accuracy"])
        eval_epochs.append(log["epoch"])

plt.figure(figsize=(8,5))
plt.plot(eval_epochs, eval_accuracy, marker="o")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Curve")

plt.savefig("speech_results_xlsr_varian/accuracy_curve.png")
plt.show()

# ----------------------------------------
# 13. Final Evaluation
# ----------------------------------------

print("\n🔥 Evaluating on Test Dataset...")

results = trainer.predict(test_dataset)

y_true = results.label_ids
y_pred = results.predictions.argmax(-1)

print("Final Test Accuracy:", accuracy_score(y_true, y_pred))

# ----------------------------------------
# 14. Confusion Matrix
# ----------------------------------------

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=languages,
    yticklabels=languages
)

plt.xlabel("Predicted Language")
plt.ylabel("True Language")
plt.title("Confusion Matrix")

plt.savefig("speech_results_xlsr_varian/confusion_matrix.png")

plt.show()